# Kaggle runner: Anatomy of a ReID Attack
**Before running:** Settings -> Accelerator = GPU T4, Internet = On. Add Input -> search 'Market-1501' and add a dataset that has `bounding_box_train`, `bounding_box_test`, `query`.
Upload this repo as a Kaggle dataset (or `git clone` it from your GitHub) and `%cd` into it.

In [ ]:
!pip -q install timm
# If you cloned from GitHub:
# !git clone https://github.com/<YOUR_USER>/reid-attack-anatomy.git && %cd reid-attack-anatomy
!python tests/smoke_test.py

## Step 1 - Train 3 victim models (≈25-40 min each on T4; run them in separate sessions if needed)

In [ ]:
DATA='/kaggle/input'
!python -m reidattack.train --data $DATA --arch resnet50 --epochs 30

In [ ]:
!python -m reidattack.train --data $DATA --arch densenet121 --epochs 30

In [ ]:
!python -m reidattack.train --data $DATA --arch vit_small --epochs 30

## Step 2 - Attack zoo + transfer matrix (evasion, then impersonation)

In [ ]:
CK='checkpoints/resnet50.pth checkpoints/densenet121.pth checkpoints/vit_small.pth'
!python scripts/run_attacks.py --data $DATA --ckpts $CK --eps 4 8 --mode evade
!python scripts/run_attacks.py --data $DATA --ckpts $CK --eps 8 --mode impersonate --attacks pgd mifgsm

## Step 3 - Universal torso patch (EOT)

In [ ]:
!python scripts/run_patch.py --data $DATA --src checkpoints/resnet50.pth --others checkpoints/densenet121.pth checkpoints/vit_small.pth --epochs 5

Download `results/` and `checkpoints/` from the Output tab, then continue with analysis (Grad-CAM, t-SNE, adversarial training).

## Step 4 - Adversarial Training Defence (ResNet50 Robustness vs Accuracy Trade-off)


In [ ]:
!python -m reidattack.adv_train --data $DATA --arch resnet50 --eps 8 --epochs 20


## Step 5 - Generate Analysis Visualizations (Heatmap, Epsilon Curves, Grad-CAM, t-SNE)


In [ ]:
!python scripts/make_baseline_table.py
!python scripts/plot_transfer_matrix.py --input results/attacks_evade.json
!python scripts/plot_eps_curve.py --input results/attacks_evade.json
!python scripts/generate_gradcam.py --ckpt checkpoints/resnet50.pth
!python scripts/plot_tsne.py
print('All analysis figures generated in results/!')


## Step 6 - Interactive Gradio Demo Verification


In [ ]:
!pip -q install gradio
!python demo/app.py
print('Gradio demo verified!')


### Pipeline Execution Complete!
Download `results/` and `checkpoints/` from the Kaggle Output tab.
